Trabajo Practico 2  
Vision por computadora I  
CEIA 2026

#### *Dependencias*

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from   scipy.fftpack     import fft2,ifft2,fftshift,ifftshift
%matplotlib qt

#### *Funcion para mostrar información del video*

In [2]:
def ShowVideoInfo(video_capture):
    if not video_capture.isOpened():
        print("Error al abrir el video.")
    else:
        # Cantidad total de frames
        total_frames = int(video_capture.get(cv2.CAP_PROP_FRAME_COUNT))

        # Fotogramas por segundo (FPS)
        fps = video_capture.get(cv2.CAP_PROP_FPS)

        # Resolución (Ancho y Alto)
        width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))

        # Códec de video (formato de compresión)
        fourcc_code = int(video_capture.get(cv2.CAP_PROP_FOURCC))
        # Convertir el código numérico a texto
        codec = (
            chr(fourcc_code & 0xFF)
            + chr((fourcc_code >> 8) & 0xFF)
            + chr((fourcc_code >> 16) & 0xFF)
            + chr((fourcc_code >> 24) & 0xFF)
        )

        # Duración estimada en segundos
        duration_seconds = total_frames / fps if fps > 0 else 0

        # Mostrar información
        print("--- INFORMACIÓN DEL VIDEO ---")
        print(f"Resolución:  {width}x{height} píxeles")
        print(f"FPS:         {fps:.2f}")
        print(f"Número de frames: {total_frames} frames")
        print(f"Códec:       {codec}")
        print(f"Duración:    {duration_seconds:.2f} segundos")

        video_capture.release()

#### *1. Algoritmo para medición de calidad de imagen*   
Pasos:
1. Calcular la FFT de la imagen de entrada (F)
2. Desplazar el espectro de F al origen ($F_c$)
3. Calcular el valor absoluto del espectro desplazado(AF) de F, donde AF = abs($F_c$)
4. Calcular el maximo valor de las componentes de frecuencia (M) de F, donde M = max(AF)
5. Calcular el numero total de px en F ($T_H$) cuyo px_value > thres, donde thres = M/1000
6. Calcular la medida de la calidad de imagen con la ecuación:
\begin{equation}
Image Quality Measure (FM) = \frac{T_H}{MxN}
\end{equation}

##### *1.1 Medición sobre todo el frame*

###### *1.1.1 Información del video*

In [3]:
video_path = 'focus_video.mov'
video_capture = cv2.VideoCapture(video_path)
ShowVideoInfo(video_capture)

--- INFORMACIÓN DEL VIDEO ---
Resolución:  640x360 píxeles
FPS:         29.97
Número de frames: 171 frames
Códec:       h264
Duración:    5.71 segundos


##### Funcion para animacion de frame y medición de enfoque

In [4]:
def Show_Frame_ISM(frame, fm, frame_counter, im_plot, hist_points, actual_point, ax2):
  '''Actualiza la interfaz gráfica 1x2 con el frame y el valor de FM recibidos'''

  # 1. Actualizamos la imagen del frame actual
  im_plot.set_data(frame)

  # 2. Recuperamos el histórico existente para añadir el nuevo dato
  x_data = list(hist_points.get_xdata())
  y_data = list(hist_points.get_ydata())

  x_data.append(frame_counter)
  y_data.append(fm)

  # 3. Actualizamos la línea azul y el punto negro destacado
  hist_points.set_data(x_data, y_data)
  actual_point.set_data([frame_counter], [fm])

  # 4. Redimensionamos los ejes automáticamente
  ax2.relim()
  ax2.autoscale_view()

  # 5. Forzamos a Matplotlib a redibujar los cambios en la ventana interactiva
  plt.draw()
  plt.pause(0.001)

###### *1.1.2 Configuración de ventana grafica e implementacion de algorimto*

In [19]:
# =====================================================================
# Configuración de interfaz gráfica
# =====================================================================
plt.ion()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Configuración del subgráfico 1 (Imagen)
ax1.set_title("Actual Frame")
ax1.axis("off")
# Creamos un fondo temporal con el tamaño estimado del video (ej. 480x640)
im_plot = ax1.imshow(np.zeros((480, 640), dtype=np.uint8), cmap="gray", vmin=0, vmax=255)

# Configuración del subgráfico 2 (Métrica)
ax2.set_title("Image Sharpness Measure")
ax2.set_xlabel("Frame")
ax2.set_ylabel("ISM")
ax2.grid(True)

# Inicializamos los objetos
(hist_points,) = ax2.plot([], [], "bo", markersize=3, alpha=0.5, label="Historic")
(actual_point,) = ax2.plot([], [], "ko", markersize=8, zorder=5, label="Actual")
ax2.legend()

plt.show(block=False)
plt.pause(0.1)

# =====================================================================
# Implementación del algoritmo de Image Quality measure
# =====================================================================
th  = 0 #numero total de pixeles que cumplen con la condicion px_value > th
fm_list = [] #lista para almacenar el fm de cada frame
frame_counter = 0

video_path = 'focus_video.mov'
video_capture = cv2.VideoCapture(video_path)

if not video_capture.isOpened():
    print("Error al abrir el archivo de video")
else:
    # Lectura frame a frame
    while True:
        ret, frame = video_capture.read()
        if ret:
            frame_counter += 1
            
            gFrame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            height, width = gFrame.shape  # Dimensiones M x N del paper
            
            # Procesamiento del frame con algoritmo FM
            # Paso 1 y 2: FFT y espectro centrado
            F = fft2(gFrame)
            Fc = fftshift(F)

            # Paso 3: Calcular AF
            AF = np.abs(Fc)

            # Paso 4: Calcular M
            M = np.max(AF)

            # Paso 5: Calcular TH
            threshold = M / 1000.0
            Th = np.sum(F > threshold)

            # Paso 6: Image Quality measure (FM) = TH / (M * N)    
            fm = Th/(width*height)
            fm_list.append(fm)
            
            Show_Frame_ISM(gFrame, fm, frame_counter, im_plot, hist_points, actual_point, ax2)
        else:
            break


# Libera el objeto de captura de video
video_capture.release()
plt.ioff()
plt.show()
cv2.destroyAllWindows()

In [18]:
# =====================================================================
# Procesamiento post-animación: Análisis de pico máximo de FM
# =====================================================================

# Listado completo de valores FM por frame
# print("\n--- LISTADO DE VALORES FM POR FRAME ---")
# for idx, val in enumerate(fm_list):
#   print(f"Frame {idx + 1}: FM = {val:.6f}")

# Encontramos el índice del frame con el valor máximo de FM
max_index = np.argmax(fm_list)
best_frame_num = max_index + 1
max_fm_valor = fm_list[max_index]

print(f"\nEl frame con mayor nitidez es el Frame #{best_frame_num}")
print(f"Valor máximo de FM alcanzado: {max_fm_valor:.6f}")

# Volvemos a extraer el frame con mayor FM para analizar su zona más nítida
video_capture = cv2.VideoCapture(video_path)
video_capture.set(cv2.CAP_PROP_POS_FRAMES, max_index)
ret_max, frame_max = video_capture.read()

if ret_max:
  gFrame_max = cv2.cvtColor(frame_max, cv2.COLOR_BGR2GRAY)
  h, w = gFrame_max.shape

  # Localizamos la zona de mayor nitidez (frecuencias altas)
  # Aplicamos la varianza del Laplaciano en bloques para encontrar la región más enfocada
  window_size = 64  # Tamaño de la ventana de análisis
  max_var = -1
  best_x, best_y = 0, 0

  # Recorremos la imagen por bloques saltando de 16 en 16 px
  for y in range(0, h - window_size, 16):
    for x in range(0, w - window_size, 16):
      block = gFrame_max[y : y + window_size, x : x + window_size]
      # Calculamos la varianza del Laplaciano en este bloque específico
      var_local = cv2.Laplacian(block, cv2.CV_64F).var()

      if var_local > max_var:
        max_var = var_local
        best_x, best_y = x, y

  # Dibujamos el Bounding Box rojo sobre el frame original a color con un grosor de 3 píxeles
  cv2.rectangle(
      frame_max,
      (best_x, best_y),
      (best_x + window_size, best_y + window_size),
      (0, 0, 255),
      3,
  )

  # Colocamos un texto indicando que es la zona de máximo enfoque
  cv2.putText(
      frame_max,
      "Zona Max FM",
      (best_x, best_y - 10),
      cv2.FONT_HERSHEY_SIMPLEX,
      0.6,
      (0, 0, 255),
      2,
  )

  # 6. Mostrar el resultado final en una ventana dedicada de OpenCV
  cv2.imshow(
      f"Frame Maximo Enfoque (#{best_frame_num}) - Zona Encontrada", frame_max)
  cv2.waitKey(0)  # Espera a que presiones cualquier tecla para cerrar

# Cierre definitivo de entornos gráficos
cv2.destroyAllWindows()
plt.ioff()
plt.show()


El frame con mayor nitidez es el Frame #114
Valor máximo de FM alcanzado: 0.114825


Al utilizar la varianza del Laplaciano analizamos que tan dispersos estan los datos con respecto a la media. Para una imagen desenfocada hay transiciones suaves, por lo tanto los valores del laplaciano son cercanos a cero y su varianza es pequeña. Para el caso de una imagen con mejor enfoque y nítida, los bordes/texturas se resaltan mas lo que hace que se tenga una mayor dispersión respecto a la media y por ende una varianza muy alta.

En lo que respecta al analisis por lotes, la FFT como tal indica las frecuencias existente pero no su ubicación en la imagen, por lo que si la dividimos por regiones y calculamos el Laplaciano para cada una, analizamos la nitidez por región y ademas al mover un objeto nitido sobre un fondo que esta desenfocado, en la región donde se haga el match obtenemos una varianza alta mientras que en los otros sera muy baja.

Por último cuando hacemos var_local > max_var, lo que buscamos son las coordenadas (x,y) de la región con bordes más definidos y las frecuencias más altas del frame.

##### *1.2 Medición sobre todo una región de interes (ROI)*

In [17]:
# =====================================================================
# Configuración de interfaz gráfica
# =====================================================================
plt.ion()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Configuración del subgráfico 1 (Imagen)
ax1.set_title("Actual Frame")
ax1.axis("off")
# Creamos un fondo temporal con el tamaño estimado del video (ej. 480x640)
im_plot = ax1.imshow(np.zeros((480, 640), dtype=np.uint8), cmap="gray", vmin=0, vmax=255)

# Configuración del subgráfico 2 (Métrica)
ax2.set_title("Image Sharpness Measure")
ax2.set_xlabel("Frame")
ax2.set_ylabel("ISM")
ax2.grid(True)

# Inicializamos los objetos
(hist_points,) = ax2.plot([], [], "bo", markersize=3, alpha=0.5, label="Historic")
(actual_point,) = ax2.plot([], [], "ko", markersize=8, zorder=5, label="Actual")
ax2.legend()

plt.show(block=False)
plt.pause(0.1)

# =====================================================================
# CONFIGURACIÓN DE LA ROI CENTRAL
# =====================================================================
# Utilizamos 0.05 para el 5% o 0.10 para el 10% del área total
ROI_AREA_RATIO = 0.10  
SCALE_FACTOR = np.sqrt(ROI_AREA_RATIO)

fm_list = []
frame_counter = 0

video_capture = cv2.VideoCapture(video_path)

if not video_capture.isOpened():
    print("Error al abrir el archivo de video")
else:
    while True:
        ret, frame = video_capture.read()
        if ret:
            frame_counter += 1
            
            gFrame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            H_total, W_total = gFrame.shape
            
            # Calculamos las dimensiones de la ROI central
            roi_w = int(W_total * SCALE_FACTOR)
            roi_h = int(H_total * SCALE_FACTOR)
            
            # Calculamos las coordenadas de inicio (esquina superior izquierda de la ROI)
            start_x = (W_total - roi_w) // 2
            start_y = (H_total - roi_h) // 2
            
            # Extraemos LA ROI (Recorte por slicing de NumPy)
            # El procesamiento se limita estrictamente a esta sub-matriz
            gFrame_roi = gFrame[start_y:start_y+roi_h, start_x:start_x+roi_w]
            
            # Procesamiento con algoritmo FM
            F = fft2(gFrame_roi)
            Fc = fftshift(F)
            
            AF = np.abs(Fc)
            M = np.max(AF)
            
            threshold = M / 1000.0
            Th = np.sum(F > threshold)
            
            # El área para normalizar es el tamaño de la ROI (roi_w * roi_h)
            fm = Th / (roi_w * roi_h)
            fm_list.append(fm)
            # =====================================================================
            
            # Dibujamos un cuadro guía de la ROI en el frame completo para visualizarlo
            # Creamos una copia en grises para mostrarla en im_plot en tiempo real
            gFrame_visualizacion = gFrame.copy()
            cv2.rectangle(gFrame_visualizacion, (start_x, start_y), 
                          (start_x + roi_w, start_y + roi_h), (255), 2)
            
            # Llamada a tu función visual pasándole el cuadro con la ROI dibujada
            Show_Frame_ISM(gFrame_visualizacion, fm, frame_counter, im_plot, 
                          hist_points, actual_point, ax2)
        else:
            break

video_capture.release()
plt.ioff()
plt.show()
cv2.destroyAllWindows()

##### *1.2 Unsharp Masking*

\begin{align}
m(x,y) &= f(x,y) - f_b(x,y) \\
g(x,y) &= f(x,y) - k \cdot m(x,y)
\end{align}

In [24]:
# =====================================================================
# Configuración de interfaz gráfica
# =====================================================================
plt.ion()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Configuración del subgráfico 1 (Imagen)
ax1.set_title("Actual Frame")
ax1.axis("off")
# Creamos un fondo temporal con el tamaño estimado del video (ej. 480x640)
im_plot = ax1.imshow(np.zeros((480, 640), dtype=np.uint8), cmap="gray", vmin=0, vmax=255)

# Configuración del subgráfico 2 (Métrica)
ax2.set_title("Image Sharpness Measure")
ax2.set_xlabel("Frame")
ax2.set_ylabel("ISM")
ax2.grid(True)

# Inicializamos los objetos
(hist_points,) = ax2.plot([], [], "bo", markersize=3, alpha=0.5, label="Historic")
(actual_point,) = ax2.plot([], [], "ko", markersize=8, zorder=5, label="Actual")
ax2.legend()

plt.show(block=False)
plt.pause(0.1)

# =====================================================================
# Implementación del algoritmo de Image Quality measure
# =====================================================================
th  = 0 #numero total de pixeles que cumplen con la condicion px_value > th
fm_list = [] #lista para almacenar el fm de cada frame
frame_counter = 0

video_path = 'focus_video.mov'
video_capture = cv2.VideoCapture(video_path)

if not video_capture.isOpened():
    print("Error al abrir el archivo de video")
else:
    # Lectura frame a frame
    while True:
        ret, frame = video_capture.read()
        if ret:
            frame_counter += 1
            
            gFrame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            height, width = gFrame.shape  # Dimensiones M x N del paper
            
            # Procesamiento del frame con algoritmo FM
            # Paso 1 y 2: FFT y espectro centrado
            F = fft2(gFrame)
            Fc = fftshift(F)

            # Paso 3: Calcular AF
            AF = np.abs(Fc)

            # Paso 4: Calcular M
            M = np.max(AF)

            # Paso 5: Calcular TH
            threshold = M / 1000.0
            Th = np.sum(F > threshold)

            # Paso 6: Image Quality measure (FM) = TH / (M * N)    
            fm = Th/(width*height)
            
            # =====================================================================
            # Unsharp Masking
            # =====================================================================
            is_sharp = False    

            # Validamos si el valor de FM está entre 0.0065 y 0.007
            if 0.0065 <= fm <= 0.007:
                # Aplicamos un desenfoque Gaussiano (kernel de 5x5)
                gauss = cv2.GaussianBlur(gFrame, (5, 5), 0)
                
                # Calculamos la máscara de enfoque y la sumamos a la original
                k = 2.0 
                gFrame_sharp = cv2.addWeighted(gFrame, 1.0 + k, gauss, -k, 0)

                new_F = fft2(gFrame_sharp)
                new_Fc = fftshift(new_F)
                new_AF = np.abs(new_Fc)
                new_M = np.max(new_AF)
                new_threshold = new_M / 1000.0
                new_Th = np.sum(new_F > new_threshold)
                fm = new_Th/(width*height)

                is_sharp = True
            
            fm_list.append(fm)

            if is_sharp:
                actual_point.set_color("red")
                actual_point.set_markersize(10)
            else:
                actual_point.set_color("black")
                actual_point.set_markersize(8)

            Show_Frame_ISM(gFrame, fm, frame_counter, im_plot, hist_points, actual_point, ax2)
        else:
            break


# Libera el objeto de captura de video
video_capture.release()
plt.ioff()
plt.show()
cv2.destroyAllWindows()